<a href="https://colab.research.google.com/github/lohithshankarappa/bisLAB/blob/main/Week5GreyWolfOptz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

# ==========================================
# 1. PREPARE THE DATASET (Mock Network Traffic Data)
# ==========================================
# 100 features total, but only some are truly useful for detecting intrusions
X, y = make_classification(n_samples=500, n_features=100, n_informative=15,
                           n_redundant=10, random_state=42)

# ==========================================
# 2. DEFINE FITNESS FUNCTION
# ==========================================
def fitness_function(position, X, y):
    """
    Evaluates a feature subset. Minimizes a blend of classification error
    and the number of features selected.
    """
    # Convert continuous position values into a binary mask (1 = selected, 0 = omitted)
    # Using standard sigmoid transformation for Binary GWO
    sigmoid = 1 / (1 + np.exp(-position))
    binary_mask = (sigmoid >= 0.5).astype(int)

    selected_indices = np.where(binary_mask == 1)[0]

    # If no features are selected by the wolf, assign a terrible fitness score
    if len(selected_indices) == 0:
        return 1.0

    # Train a quick classifier (KNN) using 3-fold cross-validation
    clf = KNeighborsClassifier(n_neighbors=3)
    scores = cross_val_score(clf, X[:, selected_indices], y, cv=3, n_jobs=-1)
    accuracy = np.mean(scores)
    error_rate = 1.0 - accuracy

    # Weight factors: 99% importance on high accuracy, 1% on minimizing feature count
    alpha_weight = 0.99
    beta_weight = 0.01
    feature_ratio = len(selected_indices) / X.shape[1]

    # Total fitness score to minimize
    fitness = (alpha_weight * error_rate) + (beta_weight * feature_ratio)
    return fitness

# ==========================================
# 3. BGWO PARAMETERS & INITIALIZATION
# ==========================================
num_wolves = 10
dimensions = X.shape[1]  # 100 features
max_iterations = 30

# Initialize positions continuously (will be converted to binary inside the fitness evaluation)
positions = np.random.uniform(-3.0, 3.0, (num_wolves, dimensions))

alpha_pos = np.zeros(dimensions)
alpha_score = float('inf')

beta_pos = np.zeros(dimensions)
beta_score = float('inf')

delta_pos = np.zeros(dimensions)
delta_score = float('inf')

# ==========================================
# 4. MAIN OPTIMIZATION LOOP
# ==========================================
print("Optimizing network features using Grey Wolf Optimizer...")

for t in range(max_iterations):
    for i in range(num_wolves):

        # Calculate fitness score for current wolf
        fitness = fitness_function(positions[i], X, y)

        # Track the top 3 best feature configurations
        if fitness < alpha_score:
            alpha_score = fitness
            alpha_pos = positions[i].copy()
        elif fitness < beta_score:
            beta_score = fitness
            beta_pos = positions[i].copy()
        elif fitness < delta_score:
            delta_score = fitness
            delta_pos = positions[i].copy()

    # Linearly decay adaptive parameter 'a' from 2 to 0
    a = 2.0 - t * (2.0 / max_iterations)

    # Update positions of all wolves based on the 3 leaders
    for i in range(num_wolves):
        for j in range(dimensions):

            # Position update formula relative to Alpha leader
            r1, r2 = np.random.rand(), np.random.rand()
            A1 = 2 * a * r1 - a
            C1 = 2 * r2
            D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
            X1 = alpha_pos[j] - A1 * D_alpha

            # Position update formula relative to Beta leader
            r1, r2 = np.random.rand(), np.random.rand()
            A2 = 2 * a * r1 - a
            C2 = 2 * r2
            D_beta = abs(C2 * beta_pos[j] - positions[i, j])
            X2 = beta_pos[j] - A2 * D_beta

            # Position update formula relative to Delta leader
            r1, r2 = np.random.rand(), np.random.rand()
            A3 = 2 * a * r1 - a
            C3 = 2 * r2
            D_delta = abs(C3 * delta_pos[j] - positions[i, j])
            X3 = delta_pos[j] - A3 * D_delta

            # Average the vectors to determine the wolf's new position entry
            positions[i, j] = (X1 + X2 + X3) / 3.0

# ==========================================
# 5. PARSE AND PRINT FINAL RESULTS
# ==========================================
# Decode the alpha wolf's optimal continuous position back into the final binary feature subset
final_sigmoid = 1 / (1 + np.exp(-alpha_pos))
final_selected_features = np.where(final_sigmoid >= 0.5)[0]

# Check the final actual classification accuracy of the optimized subset
final_clf = KNeighborsClassifier(n_neighbors=3)
final_accuracy = np.mean(cross_val_score(final_clf, X[:, final_selected_features], y, cv=3))

print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Original Feature Count : {dimensions}")
print(f"Reduced Feature Count  : {len(final_selected_features)}")
print(f"Features Dropped       : {dimensions - len(final_selected_features)}")
print(f"Indices of Selected Features:\n{final_selected_features}")
print(f"Final Model Accuracy   : {final_accuracy * 100:.2f}%")


Optimizing network features using Grey Wolf Optimizer...

=== OPTIMIZATION COMPLETE ===
Original Feature Count : 100
Reduced Feature Count  : 57
Features Dropped       : 43
Indices of Selected Features:
[ 6  7 11 13 14 17 18 21 22 23 24 25 27 30 31 32 33 34 35 36 37 40 41 42
 44 45 46 48 50 55 56 57 59 60 64 65 66 68 69 72 73 74 76 78 79 81 83 84
 86 87 90 91 94 95 97 98 99]
Final Model Accuracy   : 90.60%
